# Notebook 2: Federated Learning Experiment Sweep on IDRiD Dataset

This notebook runs the controlled federated learning experiment sweep. We compare **FedAvg** with **FedProx** across varying data heterogeneities (Dirichlet $\alpha$) and proximal regularization strengths ($\mu$) using the **EfficientNet-B0** backbone.

Based on Notebook 1 validation, we use the locked optimal hyperparameters:
- **Model**: EfficientNet-B0 (pretrained, timm)
- **Resolution**: $224 \times 224$
- **Augmentation**: ColorJitter + RandomCrop + RandomHorizontalFlip + RandomRotation
- **Local Optimizer**: AdamW (lr=$1e-4$, weight_decay=$1e-2$)
- **Rounds**: 10 rounds
- **Local Epochs**: 3 epochs per client per round
- **Batch Size**: 8

We load images dynamically from disk using our custom dataset class with 4 worker processes to speed up I/O.

## Cell 1: Environment and GPU Configuration

In [ ]:
import sys
import os
import time
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Device selected: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Max GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## Cell 2: Data Groundtruths and Paths Setup

In [ ]:
import pandas as pd

NOTEBOOK_DIR = os.getcwd()
DATA_DIR = os.path.join(NOTEBOOK_DIR, 'data', 'idrid_raw', 'B. Disease Grading')

train_csv_path = os.path.join(DATA_DIR, '2. Groundtruths', 'a. IDRiD_Disease Grading_Training Labels.csv')
test_csv_path = os.path.join(DATA_DIR, '2. Groundtruths', 'b. IDRiD_Disease Grading_Testing Labels.csv')

train_df = pd.read_csv(train_csv_path)
train_df = train_df[['Image name', 'Retinopathy grade']].dropna()
train_df['Retinopathy grade'] = train_df['Retinopathy grade'].astype(int)

test_df = pd.read_csv(test_csv_path)
test_df = test_df[['Image name', 'Retinopathy grade']].dropna()
test_df['Retinopathy grade'] = test_df['Retinopathy grade'].astype(int)

image_names = train_df['Image name'].values
labels = train_df['Retinopathy grade'].values

train_img_dir = os.path.join(DATA_DIR, '1. Original Images', 'a. Training Set')
test_img_dir = os.path.join(DATA_DIR, '1. Original Images', 'b. Testing Set')

print(f"Total Train Images: {len(train_df)}")
print(f"Total Test Images: {len(test_df)}")

## Cell 3: Dataset Class and Locked Transform Augmentations

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

class IDRiDDataset(Dataset):
    def __init__(self, image_names, labels, img_dir, transform=None):
        self.image_names = list(image_names)
        self.labels = list(labels)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.image_names)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.image_names[idx] + '.jpg')
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = int(self.labels[idx])
        return image, label

IMG_SIZE = 224
train_transform = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Datasets and Transforms successfully created.")

## Cell 4: Dirichlet Partitioner and Dataloader Setup

In [ ]:
import random
import numpy as np
from torch.utils.data import DataLoader

def partition_dirichlet_idrid(image_names, labels, num_clients, alpha, seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    
    image_names = np.array(image_names)
    labels = np.array(labels)
    num_classes = len(np.unique(labels))
    
    client_indices = [[] for _ in range(num_clients)]
    
    for class_id in range(num_classes):
        class_indices = np.where(labels == class_id)[0]
        np.random.shuffle(class_indices)
        
        proportions = np.random.dirichlet([alpha] * num_clients)
        splits = (proportions * len(class_indices)).astype(int)
        splits[-1] = len(class_indices) - splits[:-1].sum()
        
        start = 0
        for client_id, split_size in enumerate(splits):
            client_indices[client_id].extend(class_indices[start:start+split_size].tolist())
            start += split_size
            
    client_partitions = []
    for client_id in range(num_clients):
        idxs = client_indices[client_id]
        np.random.shuffle(idxs)
        client_partitions.append((image_names[idxs].tolist(), labels[idxs].tolist()))
        
    return client_partitions

print("Dirichlet partitioner loaded.")

## Cell 5: Federated Learning Utilities (BatchNorm Weight Alignment, FedProx, Evaluation)

In [ ]:
import os
import json
import torch.nn as nn
import timm

def get_weights(model):
    return [val.cpu().numpy() for val in model.state_dict().values()]

def set_weights(model, weights):
    state_dict = dict(zip(model.state_dict().keys(), [torch.tensor(w) for w in weights]))
    model.load_state_dict(state_dict, strict=True)

def train_fedprox(model, dataloader, global_weights, mu, lr, epochs, device):
    global_state_dict = dict(zip(model.state_dict().keys(), global_weights))
    global_model_params = [
        torch.tensor(global_state_dict[name]).to(device)
        for name, _ in model.named_parameters()
    ]
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    for epoch in range(epochs):
        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            batch_y = batch_y.long()
            
            optimizer.zero_grad()
            output = model(batch_x)
            ce_loss = criterion(output, batch_y)
            
            prox_term = 0.0
            for local_param, global_param in zip(model.parameters(), global_model_params):
                prox_term += torch.norm(local_param - global_param) ** 2
            
            loss = ce_loss + (mu / 2) * prox_term
            loss.backward()
            optimizer.step()
            
    return get_weights(model), len(dataloader.dataset)

def fedavg_aggregate(client_updates):
    total_samples = sum(n for _, n in client_updates)
    if total_samples == 0:
        return client_updates[0][0]
    aggregated = []
    for layer_idx in range(len(client_updates[0][0])):
        weighted_avg = sum(weights[layer_idx] * (n / total_samples) for weights, n in client_updates)
        aggregated.append(weighted_avg)
    return aggregated

def evaluate_model(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            batch_y = batch_y.long()
            output = model(batch_x)
            loss = criterion(output, batch_y)
            total_loss += loss.item() * len(batch_y)
            correct += (output.argmax(dim=1) == batch_y).sum().item()
            total += len(batch_y)
    if total == 0:
        return 0.0, 0.0
    return correct / total, total_loss / total

def compute_l2_drift(local_weights, global_weights):
    squared_sum = 0.0
    for w_local, w_global in zip(local_weights, global_weights):
        squared_sum += np.sum((w_local - w_global) ** 2)
    return np.sqrt(squared_sum)

COMPLETED_FILE = os.path.join(NOTEBOOK_DIR, 'checkpoints', 'completed.json')
def is_completed(experiment_id):
    if not os.path.exists(COMPLETED_FILE):
        return False
    try:
        with open(COMPLETED_FILE, 'r') as f:
            completed = json.load(f)
        return experiment_id in completed
    except Exception:
        return False

def mark_completed(experiment_id):
    completed = []
    os.makedirs(os.path.dirname(COMPLETED_FILE), exist_ok=True)
    if os.path.exists(COMPLETED_FILE):
        try:
            with open(COMPLETED_FILE, 'r') as f:
                completed = json.load(f)
        except Exception:
            completed = []
    completed.append(experiment_id)
    temp_file = COMPLETED_FILE + ".tmp"
    with open(temp_file, 'w') as f:
        json.dump(completed, f, indent=1)
    os.replace(temp_file, COMPLETED_FILE)
print("FL Core utilities loaded successfully.")

## Cell 6: Load Shared Evaluation Dataset (103 Test Set Images) with parallel dataloading

In [ ]:
print("[INFO] Loading shared evaluation dataset...")
test_dataset = IDRiDDataset(test_df['Image name'].values, test_df['Retinopathy grade'].values, test_img_dir, transform=eval_transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
print(f"[SUCCESS] Loaded test dataset. Size: {len(test_dataset)} images.")

## Cell 7: Single Experiment Orchestrator and Serialization

In [ ]:
def run_experiment(alpha, mu, seed, strategy, config):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        
    client_dataloaders = {}
    partitions = partition_dirichlet_idrid(image_names, labels, config['num_clients'], alpha, seed)
    
    for client_id in range(config['num_clients']):
        client_imgs, client_lbls = partitions[client_id]
        if len(client_imgs) == 0:
            client_dataloaders[client_id] = None
        else:
            client_dataset = IDRiDDataset(client_imgs, client_lbls, train_img_dir, transform=train_transform)
            # Parallel dataloading with num_workers=4 and pin_memory=True
            client_dataloaders[client_id] = DataLoader(
                client_dataset, 
                batch_size=config['batch_size'], 
                shuffle=True, 
                num_workers=4, 
                pin_memory=True, 
                drop_last=False
            )
            
    global_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5).to(device)
    global_weights = get_weights(global_model)
    
    round_metrics_list = []
    
    for round_num in range(1, config['num_rounds'] + 1):
        client_updates = []
        local_losses = []
        local_drifts = []
        
        for client_id in range(config['num_clients']):
            loader = client_dataloaders[client_id]
            if loader is None:
                client_updates.append((global_weights, 0))
                local_losses.append(0.0)
                local_drifts.append(0.0)
                continue
                
            local_model = timm.create_model('efficientnet_b0', num_classes=5).to(device)
            set_weights(local_model, global_weights)
            
            local_weights, n_samples = train_fedprox(
                model=local_model,
                dataloader=loader,
                global_weights=global_weights,
                mu=mu,
                lr=config['lr'],
                epochs=config['local_epochs'],
                device=device
            )
            
            drift = compute_l2_drift(local_weights, global_weights)
            _, client_train_loss = evaluate_model(local_model, loader, device)
            
            client_updates.append((local_weights, n_samples))
            local_losses.append(client_train_loss)
            local_drifts.append(drift)
            del local_model
            
        global_weights = fedavg_aggregate(client_updates)
        set_weights(global_model, global_weights)
        
        global_acc, global_loss = evaluate_model(global_model, test_loader, device)
        
        client_evals = []
        for client_id in range(config['num_clients']):
            loader = client_dataloaders[client_id]
            if loader is None:
                client_evals.append({'accuracy': 0.0, 'loss': 0.0})
            else:
                c_acc, c_loss = evaluate_model(global_model, loader, device)
                client_evals.append({'accuracy': c_acc, 'loss': c_loss})
                
        round_metrics_list.append({
            'round': round_num,
            'global_accuracy': global_acc,
            'global_loss': global_loss,
            'mean_train_loss': float(np.mean(local_losses)),
            'mean_drift': float(np.mean(local_drifts)),
            'max_drift': float(np.max(local_drifts)),
            'client_evals': client_evals
        })
        
        mem_allocated = torch.cuda.memory_allocated(device) / (1024 ** 2)
        print(f"  Round {round_num:2d}/{config['num_rounds']} | Global Acc: {global_acc*100:.2f}% | Global Loss: {global_loss:.4f} | GPU Mem: {mem_allocated:.1f} MB")
        
    final_acc = round_metrics_list[-1]['global_accuracy']
    best_acc = max(rm['global_accuracy'] for rm in round_metrics_list)
    final_client_accs = [c['accuracy'] for c in round_metrics_list[-1]['client_evals']]
    
    convergence = None
    target_acc = 0.95 * final_acc
    for rm in round_metrics_list:
        if rm['global_accuracy'] >= target_acc:
            convergence = int(rm['round'])
            break
            
    del global_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    return {
        'alpha': alpha,
        'mu': mu,
        'seed': seed,
        'strategy': strategy,
        'final_accuracy': final_acc,
        'best_accuracy': best_acc,
        'convergence_round': convergence,
        'accuracy_std': float(np.std(final_client_accs)) if final_client_accs else 0.0,
        'worst_client_accuracy': float(np.min(final_client_accs)) if final_client_accs else 0.0,
        'per_client_final_accuracy': final_client_accs,
        'round_metrics': round_metrics_list
    }

def save_experiment_results(result, base_dir='.'):
    round_rows = []
    for rm in result['round_metrics']:
        row = {
            'experiment_id': result['experiment_id'],
            'round': rm['round'],
            'global_accuracy': rm['global_accuracy'],
            'global_loss': rm['global_loss'],
            'mean_train_loss': rm['mean_train_loss'],
            'mean_drift': rm['mean_drift'],
            'max_drift': rm['max_drift'],
            'client_0_accuracy': rm['client_evals'][0]['accuracy'],
            'client_0_loss': rm['client_evals'][0]['loss'],
            'client_1_accuracy': rm['client_evals'][1]['accuracy'],
            'client_1_loss': rm['client_evals'][1]['loss'],
            'client_2_accuracy': rm['client_evals'][2]['accuracy'],
            'client_2_loss': rm['client_evals'][2]['loss'],
            'alpha': result['alpha'],
            'mu': result['mu'],
            'seed': result['seed'],
            'strategy': result['strategy'],
            'straggler_fraction': 0.0
        }
        round_rows.append(row)
        
    round_csv_path = os.path.join(base_dir, 'results', 'round_metrics.csv')
    os.makedirs(os.path.dirname(round_csv_path), exist_ok=True)
    round_df = pd.DataFrame(round_rows)
    cols_round = [
        'experiment_id', 'round', 'global_accuracy', 'global_loss', 'mean_train_loss',
        'mean_drift', 'max_drift', 'client_0_accuracy', 'client_0_loss', 'client_1_accuracy',
        'client_1_loss', 'client_2_accuracy', 'client_2_loss', 'alpha', 'mu', 'seed', 'strategy', 'straggler_fraction'
    ]
    round_df = round_df[cols_round]
    if os.path.exists(round_csv_path):
        round_df.to_csv(round_csv_path, mode='a', header=False, index=False)
    else:
        round_df.to_csv(round_csv_path, index=False)

    summary_csv_path = os.path.join(base_dir, 'results', 'experiment_summary.csv')
    os.makedirs(os.path.dirname(summary_csv_path), exist_ok=True)
    summary_row = {
        'experiment_id': result['experiment_id'],
        'alpha': result['alpha'],
        'mu': result['mu'],
        'seed': result['seed'],
        'strategy': result['strategy'],
        'straggler_fraction': 0.0,
        'final_accuracy': result['final_accuracy'],
        'best_accuracy': result['best_accuracy'],
        'convergence_round': result['convergence_round'],
        'accuracy_std': result['accuracy_std'],
        'worst_client_accuracy': result['worst_client_accuracy'],
        'per_client_final_accuracy': str(result['per_client_final_accuracy']),
        'runtime_seconds': result['runtime_seconds']
    }
    summary_df = pd.DataFrame([summary_row])
    cols_summary = [
        'experiment_id', 'alpha', 'mu', 'seed', 'strategy', 'straggler_fraction',
        'final_accuracy', 'best_accuracy', 'convergence_round', 'accuracy_std',
        'worst_client_accuracy', 'per_client_final_accuracy', 'runtime_seconds'
    ]
    summary_df = summary_df[cols_summary]
    if os.path.exists(summary_csv_path):
        summary_df.to_csv(summary_csv_path, mode='a', header=False, index=False)
    else:
        summary_df.to_csv(summary_csv_path, index=False)
print("Orchestration and logging functions defined.")

## Cell 8: Hyperparameter Sweep Runner Loop

In [ ]:
import time
import pandas as pd
import numpy as np

# Grid parameters
SWEEP_CONFIG = {
    'num_clients': 3,
    'num_rounds': 10,
    'local_epochs': 3,
    'batch_size': 8,
    'lr': 1e-4
}

ALPHA_VALUES = [0.1, 0.3, 1.0]
MU_VALUES = [0.0, 0.1, 1.0]
SEEDS = [42]
STRATEGIES = ['fedavg', 'fedprox']

def make_experiment_id(alpha, mu, seed, strategy):
    return f"alpha{alpha}_mu{mu}_seed{seed}_{strategy}_straggler0.0"

all_configs = []
for alpha in ALPHA_VALUES:
    all_configs.append({'alpha': alpha, 'mu': 0.0, 'strategy': 'fedavg', 'seed': 42})
    all_configs.append({'alpha': alpha, 'mu': 0.1, 'strategy': 'fedprox', 'seed': 42})
    all_configs.append({'alpha': alpha, 'mu': 1.0, 'strategy': 'fedprox', 'seed': 42})

total_configs = len(all_configs)
completed_count = 0
skipped_count = 0
runtimes = []

# Set RUN_ONLY_FIRST = False to run the full sweep grid once timing is approved
RUN_ONLY_FIRST = True

print(f"[SWEEP] Starting grid sweep: {total_configs} unique configurations defined.")
if RUN_ONLY_FIRST:
    print("[SWEEP] RUN_ONLY_FIRST mode active. Only running first config (alpha=0.3, mu=0.0, fedavg).")

for idx, conf in enumerate(all_configs):
    alpha = conf['alpha']
    mu = conf['mu']
    seed = conf['seed']
    strategy = conf['strategy']
    
    if RUN_ONLY_FIRST and not (alpha == 0.3 and mu == 0.0 and strategy == 'fedavg'):
        continue
        
    exp_id = make_experiment_id(alpha, mu, seed, strategy)
    
    if is_completed(exp_id):
        print(f"[SKIP] Experiment {exp_id} already completed.")
        skipped_count += 1
        continue
        
    print(f"\n{'='*60}")
    print(f"[RUN {completed_count + skipped_count + 1}/{total_configs}] {exp_id}")
    print(f"{'='*60}")
    
    start_time = time.time()
    try:
        res = run_experiment(
            alpha=alpha,
            mu=mu,
            seed=seed,
            strategy=strategy,
            config=SWEEP_CONFIG
        )
        elapsed = time.time() - start_time
        runtimes.append(elapsed)
        
        res['experiment_id'] = exp_id
        res['runtime_seconds'] = elapsed
        save_experiment_results(res, NOTEBOOK_DIR)
        mark_completed(exp_id)
        
        completed_count += 1
        avg_time = np.mean(runtimes)
        remaining = total_configs - (completed_count + skipped_count)
        eta = remaining * avg_time
        
        print(f"[DONE] Run finished in {elapsed:.1f}s. Final Global Accuracy: {res['final_accuracy']*100:.2f}%")
        print(f"[PROGRESS] Completed: {completed_count + skipped_count}/{total_configs} runs | ETA: {eta/60:.2f} minutes ({eta:.0f} seconds)")
    except Exception as e:
        print(f"[ERROR] Run {exp_id} failed with error: {e}")
        raise e

print(f"\n[SWEEP INCOMPLETE/COMPLETE] Executed runs in this session: {completed_count} | Skipped: {skipped_count}")